<a href="https://colab.research.google.com/github/usami0jp/ai2023-old/blob/main/rag00b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip -q install sentence-transformers transformers


In [ ]:

import torch
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. 検索
docs = [
    "物理学科のゼミは毎週水曜日の午後3時からです。",
    "大学の図書館は午前9時から午後8時まで開館しています。",
    "人工知能の授業ではPythonと機械学習を勉強します。",
    "卒業研究の発表会は2月に開催されます。"
]

# 2.
embed_model = SentenceTransformer(

    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
doc_vectors = embed_model.encode(docs, convert_to_tensor=True)

# 3. 質問

question = "物理学科のゼミは何曜日ですか？"

q_vector = embed_model.encode(question, convert_to_tensor=True)

scores = util.cos_sim(q_vector, doc_vectors)[0]

idx = torch.argmax(scores).item()

context = docs[idx]

print("検索された文書:", context)

# 4.
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(model_name)

model = model.to("cuda" if torch.cuda.is_available() else "cpu")

prompt = f"""次の資料だけを使って質問に日本語で答えてください。
資料にない場合は「わかりません」と答えてください。

資料: {context}
質問: {question}"""

messages = [{"role": "user", "content": prompt}]

text = tokenizer.apply_chat_template(

    messages, tokenize=False, add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs, max_new_tokens=100, do_sample=False
    )


answer = tokenizer.decode(
    output[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("RAGの回答:", answer)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (14 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

RAGの回答: 物理学科のゼミは毎週水曜日に開催されます。
